# PV MPPT Mission Profile → Operating Points

This notebook demonstrates how to process a PV annual irradiance/temperature profile
into MPPT operating points (voltage and current) and summarise the result as a
weighted 2-D histogram — the canonical input format for loss-map-based efficiency
calculations in `pyplecs`.

**Workflow**
1. Load the bundled `pv_annual_1kw.csv` dataset (hourly irradiance & cell temperature).
2. Filter out night-time hours (irradiance ≤ 10 W/m²).
3. Run the MPPT model (`pv_mppt`) with default 1 kW panel parameters.
4. Convert the resulting time series to a histogram of operating points.
5. Export the histogram table for downstream loss / thermal analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyplecs.mission_profile import pv_mppt, PVPanelParams, mission_profile_to_histogram

In [ ]:
from importlib import resources

data_pkg = resources.files("pyplecs.mission_profile.data")
pv_data = pd.read_csv(data_pkg / "pv_annual_1kw.csv")
pv_data.head()

In [ ]:
daytime = pv_data[pv_data["irradiance_wm2"] > 10].copy()
params = PVPanelParams()
V_mpp, I_mpp = pv_mppt(daytime["irradiance_wm2"].values, daytime["temperature_c"].values, params)
df = pd.DataFrame({"V": V_mpp, "I": I_mpp})
df["P"] = df["V"] * df["I"]
print(f"Daytime hours: {len(df)}")
print(f"V range: {df['V'].min():.1f} – {df['V'].max():.1f} V")
print(f"P range: {df['P'].min():.1f} – {df['P'].max():.1f} W")

In [ ]:
tbl = mission_profile_to_histogram(df, columns=["V", "I"], n_bins=8)

In [ ]:
print(tbl.summary())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
tbl.plot(ax=axes[0], kind="bar")
tbl.plot(ax=axes[1], kind="heatmap")
plt.tight_layout()
plt.show()

In [ ]:
tbl.to_csv("pv_mppt_ops.csv")
print("Exported to pv_mppt_ops.csv")